# Aegis Data Preparation Pipeline: Online Retail II

This notebook processes the raw `online_retail_II.xlsx` dataset into canonical parquets. This is the hardened v1.1 pipeline establishing semantically accurate foundational Customer 360 features.

## 1. Configuration

In [9]:

import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

RAW_DATA_PATH = '../../online_retail_ii/online_retail_II.xlsx'
OUTPUT_DIR = '.'

# Output files
CUSTOMERS_PARQUET = os.path.join(OUTPUT_DIR, 'customers.parquet')
PRODUCTS_PARQUET = os.path.join(OUTPUT_DIR, 'products.parquet')
TRANSACTIONS_PARQUET = os.path.join(OUTPUT_DIR, 'transactions.parquet')
FEATURES_PARQUET = os.path.join(OUTPUT_DIR, 'customer_features.parquet')
REPORT_JSON = os.path.join(OUTPUT_DIR, 'preparation_report.json')


## 2. Raw Data Loading

In [10]:

print(f"Loading raw data from {RAW_DATA_PATH}...")
xls = pd.ExcelFile(RAW_DATA_PATH)
dfs = [pd.read_excel(xls, sheet) for sheet in xls.sheet_names]
raw_df = pd.concat(dfs, ignore_index=True)
raw_row_count = len(raw_df)
print(f"Total raw rows loaded: {raw_row_count:,}")


Loading raw data from ../../online_retail_ii/online_retail_II.xlsx...
Total raw rows loaded: 1,067,371


## 3. Data Cleaning & 4. Transaction Semantics

In [11]:

df = raw_df.rename(columns={
    'Invoice': 'invoice', 'StockCode': 'product_id', 'Description': 'product_name',
    'Quantity': 'quantity', 'InvoiceDate': 'transaction_timestamp', 'Price': 'unit_price',
    'Customer ID': 'customer_id', 'Country': 'country'
}).copy()

# Remove exact duplicates
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates()

# Drop invalid prices
invalid_price_mask = df['unit_price'] < 0
df = df[~invalid_price_mask]

# Ensure strings
df['product_name'] = df['product_name'].fillna('UNKNOWN_PRODUCT').astype(str)
df['invoice'] = df['invoice'].astype(str)
df['product_id'] = df['product_id'].astype(str)

# Transaction Semantics
def get_tx_type(row):
    if str(row['invoice']).startswith('C'):
        return 'cancellation'
    elif row['quantity'] < 0:
        return 'return'
    else:
        return 'purchase'

df['transaction_type'] = df.apply(get_tx_type, axis=1)

# Financial Fields
# Purchase: positive gross, zero return.
# Return/Cancel: zero gross, positive return_amount, negative net_amount.
df['gross_amount'] = np.where(df['transaction_type'] == 'purchase', df['quantity'] * df['unit_price'], 0.0)
df['return_amount'] = np.where(df['transaction_type'] != 'purchase', abs(df['quantity']) * df['unit_price'], 0.0)
df['net_amount'] = df['gross_amount'] - df['return_amount']

# Guest Handling
df['is_guest'] = df['customer_id'].isnull()
# Keep customer_id as NaN for guests
df['customer_type'] = np.where(df['is_guest'], 'guest', 'identified')

purchase_count = (df['transaction_type'] == 'purchase').sum()
return_count = (df['transaction_type'] == 'return').sum()
cancellation_count = (df['transaction_type'] == 'cancellation').sum()
guest_tx_count = df['is_guest'].sum()


## 5. Customer & Product Normalization

In [12]:

# Identified customers only
customers = df[~df['is_guest']][['customer_id', 'country']].drop_duplicates(subset=['customer_id'], keep='first')
# Note: customer_id is float because of NaNs in the original column. We can convert to string/int for identified.
customers['customer_id'] = customers['customer_id'].astype(int).astype(str)
df.loc[~df['is_guest'], 'customer_id'] = df.loc[~df['is_guest'], 'customer_id'].astype(int).astype(str)

products = df.groupby('product_id')['product_name'].agg(lambda x: x.value_counts().index[0]).reset_index()


## 6. Transaction Normalization

In [13]:

df['transaction_line_id'] = df['invoice'] + "_" + df.groupby('invoice').cumcount().astype(str)

transactions = df[[
    'transaction_line_id', 'invoice', 'customer_id', 'product_id', 
    'transaction_timestamp', 'quantity', 'unit_price', 'gross_amount', 
    'return_amount', 'net_amount', 'transaction_type', 'country', 'is_guest'
]]


## 7. Foundational Customer 360 Features (Semantically Hardened)

In [14]:

identified_tx = transactions[~transactions['is_guest']].copy()
purchases_only = identified_tx[identified_tx['transaction_type'] == 'purchase'].copy()

# Dataset wide max purchase date for recency calculation
max_purchase_date = purchases_only['transaction_timestamp'].max()

# 1. Base aggregations over all identified tx (for financials)
agg_funcs_all = {
    'gross_amount': 'sum',
    'net_amount': 'sum',
    'return_amount': 'sum',
    'product_id': 'nunique',
    'transaction_line_id': 'count'
}
c360 = identified_tx.groupby('customer_id').agg(agg_funcs_all)
c360.columns = ['total_gross_spend', 'total_net_spend', 'total_return_amount', 'unique_products', 'total_transaction_lines']
c360 = c360.reset_index()

# 2. Purchase-specific aggregations (Dates and Invoices)
agg_funcs_purchases = {
    'transaction_timestamp': ['min', 'max'],
    'invoice': 'nunique'
}
purchase_agg = purchases_only.groupby('customer_id').agg(agg_funcs_purchases)
purchase_agg.columns = ['first_purchase_date', 'last_purchase_date', 'purchase_order_count']
purchase_agg = purchase_agg.reset_index()

c360 = c360.merge(purchase_agg, on='customer_id', how='inner') # Inner join: Customer must have at least one purchase

# 3. Line counts by type
tx_types = identified_tx.groupby(['customer_id', 'transaction_type']).size().unstack(fill_value=0)
for col in ['purchase', 'return', 'cancellation']:
    if col not in tx_types.columns: tx_types[col] = 0

tx_types = tx_types.rename(columns={
    'purchase': 'purchase_line_count',
    'return': 'return_line_count',
    'cancellation': 'cancellation_line_count'
}).reset_index()
c360 = c360.merge(tx_types[['customer_id', 'purchase_line_count', 'return_line_count', 'cancellation_line_count']], on='customer_id', how='left')

# Recency and Lifetime (Using Purchase Dates Only)
c360['recency_days'] = (max_purchase_date - c360['last_purchase_date']).dt.days
c360['customer_lifetime_days'] = (c360['last_purchase_date'] - c360['first_purchase_date']).dt.days
c360['customer_lifetime_days'] = np.maximum(c360['customer_lifetime_days'], 1) # Prevent div by zero, min is 1

# Value & Frequency
c360['average_order_value'] = np.where(c360['purchase_order_count'] > 0, c360['total_gross_spend'] / c360['purchase_order_count'], 0)
c360['purchase_frequency'] = c360['purchase_order_count'] / (c360['customer_lifetime_days'] / 30.0)

# Rates
c360['return_rate'] = c360['return_line_count'] / c360['total_transaction_lines']
c360['cancellation_rate'] = c360['cancellation_line_count'] / c360['total_transaction_lines']

# Time window features (30d, 90d from max_purchase_date)
tx_30d = identified_tx[identified_tx['transaction_timestamp'] >= (max_purchase_date - pd.Timedelta(days=30))]
tx_90d = identified_tx[identified_tx['transaction_timestamp'] >= (max_purchase_date - pd.Timedelta(days=90))]

purchases_30d = tx_30d[tx_30d['transaction_type'] == 'purchase']
purchases_90d = tx_90d[tx_90d['transaction_type'] == 'purchase']

c360 = c360.merge(tx_30d.groupby('customer_id')['net_amount'].sum().rename('net_spend_30d'), on='customer_id', how='left').fillna({'net_spend_30d': 0})
c360 = c360.merge(tx_90d.groupby('customer_id')['net_amount'].sum().rename('net_spend_90d'), on='customer_id', how='left').fillna({'net_spend_90d': 0})
c360 = c360.merge(purchases_30d.groupby('customer_id')['invoice'].nunique().rename('purchase_order_count_30d'), on='customer_id', how='left').fillna({'purchase_order_count_30d': 0})
c360 = c360.merge(purchases_90d.groupby('customer_id')['invoice'].nunique().rename('purchase_order_count_90d'), on='customer_id', how='left').fillna({'purchase_order_count_90d': 0})

customer_features = c360


## 8. Semantic Validation Gates

In [15]:

class ValidationException(Exception): pass

def assert_check(condition, msg):
    if condition: print(f"[PASS] {msg}")
    else: raise ValidationException(f"[FAIL] {msg}")

# General asserts
assert_check(customers['customer_id'].is_unique, "Customer IDs unique")
assert_check(customers['customer_id'].notnull().all(), "No null customers in canonical table")
assert_check(products['product_id'].is_unique, "Product IDs unique")

# Financial assertions
assert_check((transactions['gross_amount'] >= 0).all(), "Gross amount is non-negative")
assert_check((transactions['return_amount'] >= 0).all(), "Return amount is non-negative")
net_check = np.isclose(transactions['net_amount'], transactions['gross_amount'] - transactions['return_amount'])
assert_check(net_check.all(), "Net amount = Gross - Return")

# Semantic metric assertions
assert_check(customer_features['last_purchase_date'].notnull().all(), "No canonical customer has a null last_purchase_date")
assert_check(customer_features['first_purchase_date'].notnull().all(), "No canonical customer has a null first_purchase_date")

# AOV uses purchase order count
expected_aov = np.where(customer_features['purchase_order_count'] > 0, customer_features['total_gross_spend'] / customer_features['purchase_order_count'], 0)
assert_check(np.isclose(customer_features['average_order_value'], expected_aov).all(), "average_order_value strictly uses purchase_order_count")

# purchase frequency uses purchase order count
expected_pf = customer_features['purchase_order_count'] / (customer_features['customer_lifetime_days'] / 30.0)
assert_check(np.isclose(customer_features['purchase_frequency'], expected_pf).all(), "purchase_frequency strictly uses purchase_order_count")

assert_check((customer_features['customer_lifetime_days'] >= 1).all(), "customer_lifetime_days >= 1")

print("All semantic validations passed.")


[PASS] Customer IDs unique
[PASS] No null customers in canonical table
[PASS] Product IDs unique
[PASS] Gross amount is non-negative
[PASS] Return amount is non-negative
[PASS] Net amount = Gross - Return
[PASS] No canonical customer has a null last_purchase_date
[PASS] No canonical customer has a null first_purchase_date
[PASS] average_order_value strictly uses purchase_order_count
[PASS] purchase_frequency strictly uses purchase_order_count
[PASS] customer_lifetime_days >= 1
All semantic validations passed.


## 9. Output Generation

In [16]:

customers.to_parquet(CUSTOMERS_PARQUET, index=False)
products.to_parquet(PRODUCTS_PARQUET, index=False)
transactions.to_parquet(TRANSACTIONS_PARQUET, index=False)
customer_features.to_parquet(FEATURES_PARQUET, index=False)

report = {
    'pipeline_version': '1.2',
    'preparation_timestamp': datetime.now().isoformat(),
    'raw_row_count': raw_row_count,
    'prepared_transaction_count': len(transactions),
    'customer_count': len(customers),
    'feature_customer_count': len(customer_features),
    'product_count': len(products),
    'guest_transaction_count': int(guest_tx_count),
    'purchase_count': int(purchase_count),
    'return_count': int(return_count),
    'cancellation_count': int(cancellation_count),
    'duplicate_count_removed': int(duplicate_count),
    'feature_count': len(customer_features.columns) - 1, # minus customer_id
    'validation_results': 'ALL_PASSED'
}

with open(REPORT_JSON, 'w') as f:
    json.dump(report, f, indent=4)
print("Pipeline complete.")


Pipeline complete.
